In [1]:
%pip install -q lmdb opencv-python pillow matplotlib numpy pandas six kaggle tabulate pyzipper

import os
import subprocess
from pathlib import Path

def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "notebooks" / "data_config.ipynb").exists():
            return candidate
    return cwd.parent if cwd.name == "notebooks" else cwd

PROJECT_ROOT = find_project_root()
DOCTAMPER_DIR = PROJECT_ROOT / "DocTamper"

if not DOCTAMPER_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/qcf-568/DocTamper.git", str(DOCTAMPER_DIR)],
        check=True,
        cwd=PROJECT_ROOT,
    )

os.chdir(DOCTAMPER_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Current directory:", Path.cwd())

Note: you may need to restart the kernel to use updated packages.
PROJECT_ROOT: F:\HLCV\HLCV-Project
Current directory: F:\HLCV\HLCV-Project\DocTamper


In [3]:
%pip install -q kaggle

import json
import os
import getpass
import subprocess
import sys
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

token_path = kaggle_dir / "kaggle.json"

credential_input = input(
    "Path to kaggle.json, pasted kaggle.json content, pasted KGAT token, pasted old API key, or press Enter to type username/key: "
).strip().strip('"')

access_token_path = kaggle_dir / "access_token"
token = None

if credential_input and Path(credential_input).exists():
    token = json.loads(Path(credential_input).read_text())
elif credential_input.startswith("{"):
    token = json.loads(credential_input)
elif credential_input.startswith("KGAT_"):
    access_token_path.write_text(credential_input)
    os.chmod(access_token_path, 0o600)
    os.environ["KAGGLE_API_TOKEN"] = credential_input
elif credential_input:
    token = {
        "username": input("Kaggle username: ").strip(),
        "key": credential_input,
    }
else:
    token = {
        "username": input("Kaggle username: ").strip(),
        "key": getpass.getpass("Kaggle API key: ").strip(),
    }

if token is not None:
    token_path.write_text(json.dumps(token))
    os.chmod(token_path, 0o600)

    if not token.get("username") or not token.get("key"):
        raise ValueError("Kaggle token must contain both 'username' and 'key'.")

    print("Kaggle token saved to:", token_path)
else:
    print("Kaggle API token saved to:", access_token_path)

check = subprocess.run(
    [sys.executable, "-m", "kaggle", "datasets", "list", "-s", "doctamper"],
    text=True,
    capture_output=True,
)
if check.stdout:
    print(check.stdout)
if check.returncode != 0:
    if check.stderr:
        print(check.stderr)
    raise RuntimeError(
        "Kaggle authentication check failed. Make sure you used your Kaggle username "
        "not your email, and the API key from kaggle.json."
    )

print("Kaggle authentication works.")

Note: you may need to restart the kernel to use updated packages.
Kaggle API token saved to: C:\Users\pardi\.kaggle\access_token
ref                                title                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------  -----------------  -----------  --------------------------  -------------  ---------  ---------------  
dinmkeljiame/doctamper             DocTamper          21857296432  2024-06-04 11:06:44.310000           3090          8  0.3125           
akshitmahajan07/doctamper-dataset  DocTamper dataset   5376939503  2025-07-12 11:46:50.047000            168          4  0.3125           

Kaggle authentication works.


In [4]:
import subprocess
import sys
from pathlib import Path

DATA_DOWNLOAD_DIR = PROJECT_ROOT / "data" / "doctamper_download"
DATA_EXTRACT_DIR = PROJECT_ROOT / "data" / "doctamper_data"

DATA_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
DATA_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable,
        "-m",
        "kaggle",
        "datasets",
        "download",
        "-d",
        "dinmkeljiame/doctamper",
        "-p",
        str(DATA_DOWNLOAD_DIR),
    ],
    check=True,
)

print("Downloaded to:", DATA_DOWNLOAD_DIR)

Downloaded to: F:\HLCV\HLCV-Project\data\doctamper_download


In [5]:
import getpass
import os

os.environ["DOCTAMPER_PASSWORD"] = getpass.getpass("Paste the DocTamper unzip password from the authors' email: ")
print("Password stored temporarily.")

Password stored temporarily.


In [6]:
import os
import zipfile
from pathlib import Path

zip_path = DATA_DOWNLOAD_DIR / "doctamper.zip"
assert zip_path.exists(), f"Could not find downloaded archive: {zip_path}"

password = os.environ["DOCTAMPER_PASSWORD"].encode("utf-8")

try:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_EXTRACT_DIR, pwd=password)
except RuntimeError as exc:
    print("Python zipfile could not extract this archive, trying pyzipper...")
    import pyzipper

    with pyzipper.AESZipFile(zip_path) as zf:
        zf.pwd = password
        zf.extractall(DATA_EXTRACT_DIR)

print("Extracted to:", DATA_EXTRACT_DIR)

Extracted to: F:\HLCV\HLCV-Project\data\doctamper_data


In [7]:
mdb_files = list(DATA_EXTRACT_DIR.rglob("data.mdb"))
if not mdb_files:
    raise FileNotFoundError(f"Extraction finished, but no data.mdb was found under {DATA_EXTRACT_DIR}")

print("DocTamper data is ready.")
for mdb in mdb_files:
    print("LMDB:", mdb.parent)

DocTamper data is ready.
LMDB: F:\HLCV\HLCV-Project\data\doctamper_data\DocTamperV1-FCD
LMDB: F:\HLCV\HLCV-Project\data\doctamper_data\DocTamperV1-SCD
LMDB: F:\HLCV\HLCV-Project\data\doctamper_data\DocTamperV1-TestingSet
LMDB: F:\HLCV\HLCV-Project\data\doctamper_data\DocTamperV1-TrainingSet
